# TurboSWAN: Search for Better Settings with a Sweep

Which learning rate? Which batch size? Instead of training once per guess and keeping notes by hand, a **sweep** tries several settings in one go and records every attempt in MLflow, so you can compare them afterwards. This notebook runs a small sweep of the TurboSWAN demo model.

**What this gives you:**

- **Try many settings in one go.** You say what to vary and how many trials to run; the sweep does the rest.
- **A fair starting point.** The first trial is your recipe exactly as written, so you can see whether any other setting beats it.
- **Every trial is a full, recorded run.** Each keeps its own learning curves, settings, and fitted model, grouped under one parent run in MLflow.
- **A suggested winner, your decision.** Trials are ranked by one number, here the validation loss. The ranking tells you where to look first; it does not replace your judgment.

If you have not used MLflow or recipes before, start with `demo_tracking.ipynb`. This notebook builds on it.

## Before You Start

**Run locally:** no Data & AI platform account is needed. Follow the [getting started guide](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/demo-getting-started/) first: it shows how to get this code (branch `integration/turboswan-mlflow-sweep`), install it, choose the notebook kernel, and copy the demo data to `~/turboswan/data/demo` (access requires the Deltares-research organization). Then make sure the notebook's working directory is `projects/turboswan/notebooks` and run the cells in order.

**Platform tracking is separate from platform compute.** The sweep can record its parent run, trial metrics, plots and fitted models in the Data & AI platform's MLflow while training runs on your own machine. Set `run.mode = "dnai"` in a copy of the sweep's base recipe, remove `mlflow.local_tracking_dir` if present, and point the copied sweep file's `base_config_toml` at that recipe. The driver and every trial worker need network access to platform MLflow and its artifact storage, with the required credentials supplied through their environment; these are not automatically provided on your own machine. Data still comes from an existing folder on the training machine. This records new runs there; it does not upload earlier local runs.

**Remote platform execution is not yet wired up for this notebook.** Its `ray.init(...)` starts Ray locally in the usual setup; choosing `dnai` changes tracking, not job placement. Running it inside JupyterHub would still use that server's resources, not the platform's remote Ray workers, and the parallel demo may exceed the small server's resources. Remote workers need a separate submission setup with the code, dependencies and data available to them. This notebook does not configure that or submit jobs to Snellius. For the single-training JupyterHub route, see [Prepare JupyterHub](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/demo-platform-setup/) and [Run the tracked demo](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/demo-mlflow-platform/) (new pages pending publication; these are not remote-sweep instructions).

Every choice in this notebook comes from small text files (TOML). Keep the [TOML configuration guide](https://potential-doodle-8gm7w41.pages.github.io/turboswan-mlx/) open next to the notebook: its page on the sweep file explains each setting (access requires the Deltares-research organization).

A sweep trains several models, so it takes longer than a single run: a few minutes for this demo. The small model and short training schedule demonstrate the workflow; they are not a recommendation for a scientific experiment.

In [ ]:
import logging
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

import ray

from turboswan import training
from turboswan.mlops.sweeps.configuration import load_sweep
from turboswan.mlops.sweeps.blocking.driver import run_sweep
from turboswan.mlops.sweeps.blocking.executors import RayExecutor

# progress messages show which trial has finished
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

## 1. Choose a Sweep

A sweep uses two small text files. The **sweep file** says which recipe to start from, what to vary, and how many trials to run. The **recipe** is the one you know from the tracking notebook. Copy the sweep file before changing anything and point `SWEEP_FILE` below to your copy. Run the next cell and read the file it prints; the comments explain each line.

In [ ]:
SWEEP_FILE = Path("../configs/sweeps/demo.toml")

sweep, strategy, search_space = load_sweep(SWEEP_FILE)

print(SWEEP_FILE.read_text())

### The recipe it starts from

Every trial is this recipe with the searched values swapped in. So the sweep wins over the recipe: a `learning_rate` you read below is replaced in every trial except the first, which uses the recipe as written (when `include_baseline = true`). `[mlflow]` names the study the sweep is recorded under. The data and model sections are explained in the tracking notebook.

In [ ]:
BASE_RECIPE = SWEEP_FILE.parent / sweep.base_config_toml

recipe = training.load_config(BASE_RECIPE)
# the same recipe as plain settings, which the sweep copies and overrides for each trial
base_config, recipe_toml = training.load_config_source(BASE_RECIPE)

training.show_recipe_section(recipe, "training")
print()
training.show_recipe_section(recipe, "mlflow")

## 2. Optional: Watch the Trials Live

Open MLflow before the sweep starts. Run the next cell. It starts MLflow in the background; after a few seconds, open http://127.0.0.1:5001 in your browser. MLflow keeps running until you close this notebook's kernel or VS Code, so you only need to run the cell once.

Once the sweep starts, open your study (the `experiment_name` printed above). It shows one parent run named `sweep-` followed by the start time, with the trials listed as its child runs. Refresh the page to see new trials appear, and the loss charts of the running trial grow after each epoch.

In [ ]:
if recipe.run.mode == "local":
    record_folder = (recipe.mlflow.local_tracking_dir or Path.home() / "turboswan" / "mlflow").expanduser().resolve()
    record_folder.mkdir(parents=True, exist_ok=True)
    subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--port", "5001", "--backend-store-uri", f"sqlite:///{record_folder / 'mlflow.db'}"])
    print("MLflow is starting. In a few seconds, open http://127.0.0.1:5001 in your browser.")
else:
    print("This recipe records to the Data & AI platform's MLflow. Open it there.")

## 3. Run the Sweep

The next cell trains every trial one after another, each as a complete tracked run. **Ray**, a tool that runs work as separate processes on your own machine, manages the trials: a local Ray instance starts when the cell runs and is shut down afterwards. This takes a few minutes. Progress messages show which trial has finished.

In [ ]:
ray.init(ignore_reinit_error=True)

try:
    result = run_sweep(
        sweep,
        strategy,
        search_space,
        base_config,
        sweep_id=f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}",
        recipe_toml=recipe_toml,
        # each trial gets 2 CPU cores and no GPU; raise this if your machine has room
        executor=RayExecutor(cpus_per_trial=2, gpus_per_trial=0),
    )
finally:
    ray.shutdown()

## 4. Read the Results

Each trial is ranked by one number: the value of `selection_metric` from the sweep file after its last epoch. The next cell lists what each trial changed and how it scored, and marks the best one.

In [ ]:
label = recipe.mlflow.metric_labels.get(sweep.selection_metric, sweep.selection_metric)
better = "lower" if sweep.direction == "minimize" else "higher"
print(f"Ranked by: {label} ({better} is better)\n")

for trial in result["results"]:
    changed = {key: value for key, value in trial["overrides"].items() if key != "run.model_version"}
    print(f"Trial {trial['config_index']}: {changed or 'recipe as written'}")
    if "error" in trial:
        print(f"    failed: {trial['error']}")
        continue
    best = "  <- best" if trial["config_index"] == result["winner"]["config_index"] else ""
    print(f"    {label}: {trial[sweep.selection_metric]:.4f}{best}")

## 5. Compare in MLflow

The list above is a summary; MLflow has the full record. Open your study and:

- open the parent run `sweep-...`: it keeps the sweep settings, the search space, the best value, and the winning trial (`winner_run_id`);
- select the trials and choose **Compare** to overlay their learning curves and see which settings differ;
- tag the trials worth following up on, as in the tracking notebook.

The demo sweep shows the mechanics. Three very short trials on a few data files say little about which setting is better for your science, and the validation loss is measured on a demo split, not an independent evaluation.

## 6. Try Your Own Search

Change what the sweep tries by editing your copy of the sweep file and saving it. For example:

- raise `max_configs` to `5` for more trials;
- change the range of `training.learning_rate`, or add a line such as `"training.n_epochs" = [3, 5]` under `[search_space]`;
- change `seed` to get different random draws.

Then rerun from the cell that loads the sweep file (`load_sweep`) down to the cell that runs it. The files are read again when the cells run, so you do not need to restart the notebook. If you edit the recipe instead, rerun from the cell that loads it. Refresh MLflow: the new sweep appears as another parent run next to the first, and the old one keeps its own record.